In [2]:
# Find energy indices whose condition number exceeds a threshold.
import h5py
import numpy as np

h5path = "/scratch/yimili/matrices/hdf5/si-bulk.h5"   # <-- set your material
threshold = 1e4

with h5py.File(h5path, "r") as f:
    kappa = f["global/condition_full_svd"][:]
    indices = f["metadata/indices"][:] if "metadata/indices" in f else np.arange(len(kappa))

    mask = kappa > threshold
    bad_indices = indices[mask]
    bad_kappa = kappa[mask]

    print(f"{mask.sum()} / {len(kappa)} indices have condition number > {threshold:.0e}")

    for idx, k in zip(bad_indices, bad_kappa):
        rhs_path = f"E_{idx}/rhs"
        if rhs_path in f:
            rhs = f[rhs_path]
            rhs_shape = rhs.shape
            rhs_dtype = rhs.dtype
            # norm requires loading the data -- fine for a quick sanity check,
            # skip this line if rhs is huge and you just want shape/dtype
            rhs_norm = np.linalg.norm(rhs[:])
            print(f"  idx={idx:<5}  kappa={k:.3e}  rhs.shape={rhs_shape}  "
                  f"rhs.dtype={rhs_dtype}  ||rhs||={rhs_norm:.3e}")
        else:
            print(f"  idx={idx:<5}  kappa={k:.3e}  rhs: NOT FOUND ({rhs_path} missing)")

bad_indices   # last expression -> shown as cell output too

111 / 402 indices have condition number > 1e+04
  idx=7      kappa=1.840e+04  rhs.shape=(3840, 10)  rhs.dtype=complex128  ||rhs||=7.805e+00
  idx=17     kappa=1.431e+04  rhs.shape=(3840, 10)  rhs.dtype=complex128  ||rhs||=7.853e+00
  idx=26     kappa=1.311e+04  rhs.shape=(3840, 10)  rhs.dtype=complex128  ||rhs||=7.895e+00
  idx=190    kappa=1.271e+04  rhs.shape=(3840, 13)  rhs.dtype=complex128  ||rhs||=9.458e+00
  idx=199    kappa=1.492e+04  rhs.shape=(3840, 12)  rhs.dtype=complex128  ||rhs||=9.080e+00
  idx=201    kappa=6.845e+04  rhs.shape=(3840, 10)  rhs.dtype=complex128  ||rhs||=8.623e+00
  idx=206    kappa=2.668e+04  rhs.shape=(3840, 10)  rhs.dtype=complex128  ||rhs||=8.775e+00
  idx=210    kappa=1.337e+04  rhs.shape=(3840, 11)  rhs.dtype=complex128  ||rhs||=9.000e+00
  idx=215    kappa=2.513e+04  rhs.shape=(3840, 10)  rhs.dtype=complex128  ||rhs||=9.312e+00
  idx=218    kappa=1.112e+04  rhs.shape=(3840, 10)  rhs.dtype=complex128  ||rhs||=9.116e+00
  idx=219    kappa=1.749e+04  rh

array([  7,  17,  26, 190, 199, 201, 206, 210, 215, 218, 219, 220, 224,
       229, 232, 233, 234, 236, 239, 242, 246, 249, 250, 252, 254, 257,
       260, 264, 267, 269, 270, 271, 272, 273, 274, 275, 278, 279, 280,
       287, 288, 289, 293, 294, 295, 296, 297, 309, 310, 311, 318, 319,
       320, 321, 322, 323, 324, 325, 326, 327, 328, 338, 339, 340, 341,
       342, 343, 344, 345, 346, 347, 348, 349, 350, 351, 352, 353, 354,
       355, 356, 357, 358, 359, 360, 361, 362, 363, 364, 365, 366, 367,
       368, 369, 370, 371, 372, 373, 374, 375, 376, 377, 378, 379, 380,
       381, 382, 391, 392, 393, 400, 401])

In [1]:
import h5py

path = '/scratch/yimili/matrices/hdf5/carbon-chain.h5'

with h5py.File(path, 'r') as f:
    def show(name, obj):
        kind = "GROUP" if isinstance(obj, h5py.Group) else "DATASET"
        extra = f"shape={obj.shape} dtype={obj.dtype}" if isinstance(obj, h5py.Dataset) else ""
        print(f"{kind:7s} {name}  {extra}")
    f.visititems(show)

GROUP   E_0  
GROUP   E_0/M  
DATASET E_0/M/data  shape=(246094,) dtype=complex128
DATASET E_0/M/indices  shape=(246094,) dtype=int32
DATASET E_0/M/indptr  shape=(2601,) dtype=int32
GROUP   E_0/Sigma  
DATASET E_0/Sigma/data  shape=(13778,) dtype=complex128
DATASET E_0/Sigma/indices  shape=(13778,) dtype=int32
DATASET E_0/Sigma/indptr  shape=(2601,) dtype=int32
GROUP   E_0/blockthomas  
GROUP   E_0/blockthomas/complex128  
DATASET E_0/blockthomas/complex128/Dmod_lu  shape=(25, 104, 104) dtype=complex128
DATASET E_0/blockthomas/complex128/Dmod_piv  shape=(25, 104) dtype=int32
DATASET E_0/blockthomas/complex128/L  shape=(24, 104, 104) dtype=complex128
DATASET E_0/blockthomas/complex128/U  shape=(24, 104, 104) dtype=complex128
DATASET E_0/blockthomas/complex128/time_fact  shape=() dtype=float64
DATASET E_0/blockthomas/complex128/time_solve  shape=() dtype=float64
DATASET E_0/blockthomas/complex128/x  shape=(2600, 4) dtype=complex128
GROUP   E_0/blockthomas/complex64  
DATASET E_0/blocktho

In [6]:
#!/usr/bin/env python3
"""
Delete leftover "solvers" groups (from the old E_<idx>/solvers/<solver>/<dtype>
layout) from a material's HDF5 file. Current layout saves solvers as direct
siblings of M/rhs/Sigma (E_<idx>/superlu/...), so any E_<idx>/solvers group
still present is stale data from before that change.

Usage:
    python clean_solvers_group.py /path/to/carbon-nanotube.h5
"""
# ---- Cell 1: library import----
import numpy as np
import matplotlib.pyplot as plt
import scipy.sparse as sp
import h5py
from pathlib import Path

hdf5_dir = Path("/scratch/yimili/matrices/hdf5")
name     = "si-bulk"          # change per notebook
material = name 
h5path   = hdf5_dir / f"{name}.h5"


with h5py.File(h5path, "a") as f:
    to_delete = [f"{key}/solvers" for key in f.keys()
                if key.startswith("E_") and "solvers" in f[key]]

    print(f"Found {len(to_delete)} leftover 'solvers' group(s).")
    for path in to_delete:
        print(f"  deleting {path}")
        del f[path]

print("Done.")

Found 0 leftover 'solvers' group(s).
Done.
